#  Guía básica para  proyecto Transcriptómica



Abrir consola

In [ ]:
En el buscador de Windows escribir "cmd" o buscar la consola  

#### Ingreso usuario

In [ ]:
ssh camilo@10.100.27.2
password: c4m1lo
ssh camila@10.100.27.2
password: c4m1la
ssh 4dm1n@10.100.27.2
password: 882664
ssh fmorroni@10.100.27.2
password: 20389687


#### Activar el entorno

In [ ]:
Para camilo
conda activate bioenv2 

Para 4dm1nn
conda activate bioenv2


Para fmorroni     #¿Cómo puedo hacer que se active automáticamente el ambiente bioenv2 al iniciar sesión en la terminal?
$ source ~/miniconda3/etc/profile.d/conda.sh
$ conda activate bioenv2
(bioenv2) [fmorroni@gnomaciencias ~]$


Para camila
$ source ~/miniconda3/etc/profile.d/conda.sh
$ conda actívate bioenv2 
(bioenv2) [camila@gnomaciencias ~]$ 

cd /data/Labyger  #Ruta directa a la carpeta Labyger

#### Comandos básicos CMD

In [ ]:


Windows + R y escribir "cmd" para abrir la consola de comandos en Windows.
pwd        # muestra en qué carpeta estás
ls         # lista archivos
cd carpeta # moverse entre carpetas en carpeta debes colocar el nombre
cd ..      # subir un nivel o salirse de la carpeta en la que estas
mkdir dir  # crear carpeta luego del dir viene el nombre de la carpeta
mkdir -p dir/subdir # crear carpeta y subcarpeta
rm archivo # borrar archivo
cp a b     # copiar
mv a b     # mover/renombrar
mkdir ~/directoriocolaborativo # crear una carpeta en el home del usuario, luego del ~ viene el nombre de la carpeta
groups     # muestra a qué grupos pertenece el usuario
sudo -1      # muestra los comandos que el usuario puede ejecutar con sudo
whoami       # muestra el nombre del usuario con el que estás logueado
pwd         # muestra en qué carpeta estás
id         # muestra el ID del usuario y los grupos a los que pertenece
cd /        # ir a la raíz del sistema de archivos
top        # muestra los procesos en ejecución y el uso de recursos del sistema
htop       # una versión mejorada de top, con una interfaz más amigable (pu
q  #quit sirve para salir de top o htop



#### Comandos básicos Visual Studio Code

In [ ]:


Ctrl + Ñ para abrir la terminal en Linux, MacOs y Visual Studio Code.


#### Comandos editor de texto cmd



In [ ]:
^G Help        # ayuda  
^C Cancel      # cancelar guardar  
M-D DOS Format # guardar formato Windows  
M-M Mac Format # formato Mac  
M-A Append     # agregar al final del archivo  
M-P Prepend    # agregar al inicio  
^T Browse      # buscar archivo  

#### Comandos básicos 4dm1n


In [ ]:
sudo -i  # iniciar sesión como root (administrador)
sudo -l  # muestra los comandos que el usuario puede ejecutar con sudo
adduser  # crear un nuevo usuario
usermod -aG sudo nombre_usuario  # agregar un usuario al grupo sudo para que tenga permisos de administrador
passwd nombre_usuario  # cambiar la contraseña de un usuario
chown root:grupo archivo  # cambiar el propietario y el grupo de un archivo
chmod 755 archivo  # cambiar los permisos de un archivo (rwxr-xr-x)


#### Comandos básicos FastQc



In [ ]:


head archivo.fastq   # ver primeras líneas
less archivo.fastq   # ver archivo completo
wc -l archivo.fastq  # contar líneas
*.fastq              # todos los archivos fastqc

## Uso de programas bioinformaticos

## 1) FastQc -> Control de calidad inicial
Objetivo: evaluar la calidad de las lecturas crudas (FASTQ) antes de cualquier filtrado. 

Nota: Es el único programa que usaremos en modo visual, no es necesario buscar las carpetas con comandos.

In [ ]:
# Crear carpeta para resultados
mkdir fastqc_raw

# Ejecutar FastQC
fastqc

#File
#Elegir archivos (Máximo 2 al mismo tiempo)
#Guardar archivos en la carpeta creada ateriormente

Una vez terminado, abre los archivos .html generados en fastqc_raw con un navegador.
# Te mostrarán:
# - Calidad promedio por base (idealmente > Q30)
# - Contaminación por adaptadores
# - Distribución de contenido GC
# - Longitud de secuencias


## 2) Trimmomatic -> Filtrado y recorte de lecturas
Objetivo: Se usa el programa Trimmomatic, que tiene distintos parámetros ajustables según el tipo de secuenciación (aquí usamos paired-end).

Tool: Trimmomatic v0.39

In [ ]:
# Ejecutar Trimmomatic en modo paired-end (lecturas hacia adelante y reversas)
trimmomatic PE \
  -threads                                           # Número de hilos a usar (CPU que se utilizarán)
  -phred33                                           # Especifica la codificación del puntaje de calidad. phred33 es el estándar para los datos modernos de Illumina.
  -trimlog                                           # Escribe un archivo de registro detallado (log)
  trim_sample_R1.log                                 # Nombre del archivo de registro detallado (log)
  sample_R1.fastq.gz sample_R2.fastq.gz              # Archivos FASTQ de entrada
  sample_R1_paired.fq.gz sample_R1_unpaired.fq.gz    # Lecturas "forward" (emparejadas y no emparejadas)
  sample_R2_paired.fq.gz sample_R2_unpaired.fq.gz    # lecturas "reverse" (emparejadas y no emparejadas)
  baseout                                            # Código para archivos de salida
  R1_trimmed.fq.gz                                   # Nombre archivo resultante post trimming
  ILLUMINACLIP:TruSeq3-PE.fa:2:30:10                 # Archivo de adaptadores y parámetros de recorte
  LEADING:20                                         # Recorta bases con calidad <20 al inicio desde 5'
  TRAILING:20                                        # Recorta bases con calidad <20 al final desde 3'
  SLIDINGWINDOW:4:15                                 # Recorta cuando el promedio de calidad en 4 bases es <2015
  MINLEN:150                                         # Descarta lecturas menores a 120 bases


Código que usamos:

  trimmomatic PE \-threads 20 \-phred33 \-trimlog ../../data/trimmomatic/trim_GM_2.log \
  ../../data/C_rogercresseyi/GM_2_1.fq.gz ../../data/C_rogercresseyi/GM_2_2.fq.gz \
  -baseout ../../data/trimmomatic/GM_2_trimmed.fq.gz \ILLUMINACLIP:$CONDA_PREFIX/share/trimmomatic/adapters/
  TruSeq3-PE.fa:2:30:10 \LEADING:20 \TRAILING:20 \SLIDINGWINDOW:4:15 \MINLEN:120


## 3) SortmeRNA -> Depleción del ARNr
Objetivo: Se usa el programa SortmeRNA  para separar las lecturas tanto del host (Ribosomal), como las del microbioma

Tool: SortMeRNA v4.3.6

Databases: SILVA, Rfam

In [ ]:

# Ejecutar SortmeRNA con las lecturas pares (sample_R1_paired.fq.gz)

#Bases de datos de referencia
--ref /media/metatrans/HDD/sortmerna/silva-euk-28s-id98.fasta          #Eucariota 28s
--ref /media/metatrans/HDD/sortmerna/silva-euk-18s-id95.fasta          #Eucariota 18s
--ref /media/metatrans/HDD/sortmerna/silva-bac-23s-id98.fasta          #Bacteria 23s
--ref /media/metatrans/HDD/sortmerna/silva-bac-16s-id90.fasta          #Bacteria 16s
--ref /media/metatrans/HDD/sortmerna/silva-arc-23s-id98.fasta          #Archea  23s
--ref /media/metatrans/HDD/sortmerna/silva-arc-16s-id95.fasta          #Archea 16s
--ref /media/metatrans/HDD/sortmerna/rfam-5s-database-id98.fasta       #rfam 5s
--ref /media/metatrans/HDD/sortmerna/rfam-5.8s-database-id98.fasta     #rfam 5.8s

#Archivos trimomatic Pares (Paired end) -> Forward y reverse ya filtrado de trimommatic
--reads /trimmomatic/*_1P.fq
--reads /trimmomatic/*_2P.fq

#Salida de lecturas no ribosomales
--other /sortmerna/*_smr.fq     #indica el archivo de salida donde se guardarán las lecturas que NO coinciden con rRNA

#Formato y estructura de salida
--out2       #Genera dos archivos de salida por cada input emparejado, uno para read1 y otro para read2
--fastx      #Guarda los resultados en formato FASTQ (con calidad), en vez de FASTA.

#Rendimiento
--threads 24 #En este caso el comando usa 24 hilos de CPU

#Directorio de trabajo temporal
--workdir /smrwd/* #Donde SortMeRNA guardará sus archivos temporales e índices.

#Modo de lecturas emparejadas
--paired_in   #modo paired-end, es decir, que _1P.fq y _2P.fq son pares.

Command example:

sortmerna --ref /media/metatrans/HDD/sortmerna/silva-euk-28s-id98.fasta --ref 
/media/metatrans/HDD/sortmerna/silva-euk-18s-id95.fasta --ref /media/metatrans/HDD/sortmerna/silva-bac-23s-id98.fasta --
ref /media/metatrans/HDD/sortmerna/silva-bac-16s-id90.fasta --ref /media/metatrans/HDD/sortmerna/silva-arc-23s-
id98.fasta --ref /media/metatrans/HDD/sortmerna/silva-arc-16s-id95.fasta --ref /media/metatrans/HDD/sortmerna/rfam-5s-
database-id98.fasta --ref /media/metatrans/HDD/sortmerna/rfam-5.8s-database-id98.fasta --reads /trimmomatic/*_1P.fq --
reads /trimmomatic/*_2P.fq --other /sortmerna/*_smr.fq --out2 --fastx --threads 24 --workdir /smrwd/* --paired_in 


Este codigo usamos en Calygus:

